# Tax Optimization — Complete Ground-Up Guide
### For quant developer interviews | CS background assumed, German/EU tax law built from scratch

---

**What this notebook covers:**

1. Why tax matters in wealth management — the compounding deferral effect
2. German capital gains tax structure (Kapitalertragsteuer, Vorabpauschale, Freistellungsauftrag)
3. European tax landscape for pan-European robo-advisors
4. Tax lot accounting — FIFO, LIFO, specific identification, cost basis methods
5. Optimal lot selection as a linear program
6. Tax-loss harvesting — the deferral value calculation in full detail
7. Wash-sale equivalents in Germany and EU jurisdictions
8. Tax-aware rebalancing — integrating tax into the convex optimizer
9. Vorabpauschale — the unique German annual pre-payment tax
10. Production implementation patterns

**Dependencies:**
```
pip install numpy scipy matplotlib cvxpy
```

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog, minimize
from datetime import date, timedelta
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

try:
    import cvxpy as cp
    CVXPY = True
except ImportError:
    CVXPY = False
    print('cvxpy not found — tax-aware optimizer will use scipy fallback')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f8f8',
    'axes.grid': True, 'grid.color': 'white', 'grid.linewidth': 0.8,
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
})

PURPLE = '#534AB7'; TEAL = '#1D9E75'; AMBER = '#EF9F27'
CORAL  = '#D85A30'; BLUE = '#185FA5'; GREEN = '#3B6D11'
RED    = '#A32D2D'; GRAY = '#888780'

# ── German tax constants ─────────────────────────────────────────────────────
KAPITALERTRAGSTEUER = 0.25          # 25% flat capital gains tax
SOLIDARITAETSZUSCHLAG = 0.055       # 5.5% soli on the tax amount
EFFECTIVE_TAX_RATE = KAPITALERTRAGSTEUER * (1 + SOLIDARITAETSZUSCHLAG)  # 26.375%
FREISTELLUNGSAUFTRAG_SINGLE  = 1000  # €1,000 annual exemption (single)
FREISTELLUNGSAUFTRAG_MARRIED = 2000  # €2,000 annual exemption (married)
BASISZINS_2024 = 0.0256             # Basiszins for Vorabpauschale 2024
BASISERTRAG_FACTOR = 0.70           # 70% of Basiszins is the Basisertrag
TEILFREISTELLUNG_EQUITY = 0.30      # 30% exempt for equity funds
TEILFREISTELLUNG_MIXED  = 0.15      # 15% exempt for mixed funds

print(f'German effective tax rate: {EFFECTIVE_TAX_RATE:.4f} = {EFFECTIVE_TAX_RATE*100:.3f}%')
print(f'Tax constants loaded.')

German effective tax rate: 0.2637 = 26.375%
Tax constants loaded.


---
## Part 1 — Why Tax Matters: The Compounding Deferral Effect

Tax optimization in wealth management is not about avoiding taxes — it is about **deferral**. Every euro of tax you pay today is a euro that stops compounding. Every euro of tax payment deferred continues growing in the portfolio.

### The deferral math

Consider an investor who has a €10,000 unrealized gain in a position. They can:
- **Sell now:** pay €2,637.50 in tax (26.375%), leaving €7,362.50 invested
- **Defer 5 years:** keep the full €10,000 + gain invested, pay tax later

The present value of paying tax T years from now vs today, at discount rate $r$:
$$\text{PV savings} = \text{tax} \times (1 - (1+r)^{-T}) \approx \text{tax} \times r \times T \text{ (for small r)}$$

This is an **interest-free loan** from the tax authority equal to the tax amount, for T years.

### The three levers of tax optimization in a robo-advisor
1. **Lot selection:** When you must sell, choose the lots with the highest cost basis first — minimising the realised gain and thus the immediate tax bill.
2. **Tax-loss harvesting (TLH):** Proactively sell positions with *unrealised losses* to realise those losses, which offset gains elsewhere — accelerating the tax refund.
3. **Holding period management:** Some jurisdictions give preferential rates for long-term holdings. Germany does not (flat rate regardless of holding period), but this is critical for US clients and some EU countries.

In [ ]:
# ── Compounding deferral benefit: full quantification ────────────────────────
def deferral_benefit(gain_eur, tax_rate, growth_rate, defer_years, reinvest=True):
    """
    Compare: pay tax now vs defer T years.
    Returns the extra terminal wealth from deferring.
    """
    # Scenario A: pay tax now, invest remaining
    tax_now    = gain_eur * tax_rate
    invest_now = gain_eur - tax_now
    terminal_A = invest_now * (1 + growth_rate)**defer_years

    # Scenario B: defer tax, invest full amount, pay tax at end
    grow_full  = gain_eur * (1 + growth_rate)**defer_years
    extra_gain = grow_full - gain_eur
    tax_future = (gain_eur + extra_gain) * tax_rate  # tax on original + new gains
    terminal_B = grow_full - tax_future

    return terminal_B - terminal_A, terminal_A, terminal_B

# Parameters
gain      = 10_000  # €10,000 unrealised gain
tax_rate  = EFFECTIVE_TAX_RATE
r         = 0.07    # 7% annual growth

years     = np.arange(1, 31)
benefits  = [deferral_benefit(gain, tax_rate, r, y)[0] for y in years]
terminal_A_arr = [deferral_benefit(gain, tax_rate, r, y)[1] for y in years]
terminal_B_arr = [deferral_benefit(gain, tax_rate, r, y)[2] for y in years]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Terminal wealth comparison
axes[0].plot(years, terminal_A_arr, color=CORAL,  lw=2.5, label='Pay tax now, invest rest')
axes[0].plot(years, terminal_B_arr, color=TEAL,   lw=2.5, label='Defer tax, invest full')
axes[0].fill_between(years, terminal_A_arr, terminal_B_arr, alpha=0.2, color=TEAL, label='Deferral benefit')
axes[0].set_xlabel('Years of deferral')
axes[0].set_ylabel('Terminal wealth (€)')
axes[0].set_title(f'Terminal wealth: pay now vs defer\nGain = €{gain:,}, r={r*100:.0f}%, tax={tax_rate*100:.2f}%')
axes[0].legend(fontsize=9)
for y in [5, 10, 20]:
    b = deferral_benefit(gain, tax_rate, r, y)[0]
    axes[0].annotate(f'{y}yr: +€{b:.0f}', (y, terminal_B_arr[y-1]),
                     textcoords='offset points', xytext=(5, 5), fontsize=8, color=TEAL)

# Deferral benefit (€) over time
axes[1].plot(years, benefits, color=GREEN, lw=2.5)
axes[1].fill_between(years, 0, benefits, alpha=0.2, color=GREEN)
axes[1].set_xlabel('Years of deferral')
axes[1].set_ylabel('Extra terminal wealth from deferral (€)')
axes[1].set_title('Deferral value grows non-linearly\n(compounding on the deferred tax works for you)')

# Sensitivity to growth rate
growth_rates = [0.03, 0.05, 0.07, 0.10, 0.12]
colors_g = [BLUE, TEAL, GREEN, AMBER, CORAL]
for gr, col in zip(growth_rates, colors_g):
    bens = [deferral_benefit(gain, tax_rate, gr, y)[0] for y in years]
    axes[2].plot(years, bens, color=col, lw=2, label=f'r={gr*100:.0f}%')
axes[2].set_xlabel('Years of deferral')
axes[2].set_ylabel('Deferral benefit (€)')
axes[2].set_title('Deferral benefit by portfolio growth rate\nHigher growth → more valuable to defer')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

for y in [1, 5, 10, 20, 30]:
    b, tA, tB = deferral_benefit(gain, tax_rate, r, y)
    pv_approx = gain * tax_rate * r * y
    print(f'  {y:2d} years: benefit = €{b:,.0f}  (approx PV formula: €{pv_approx:,.0f})')

---
## Part 2 — German Capital Gains Tax: Complete Reference

Scalable Capital is a German-licensed investment platform. Understanding German tax law is explicit in the job description.

### Kapitalertragsteuer (KESt) — the flat capital gains tax
Germany taxes capital income (dividends, interest, capital gains) at a flat rate:
- **Base rate:** 25%
- **Solidaritätszuschlag (Soli):** 5.5% of the 25% → effective 1.375%
- **Total effective rate:** 26.375%
- **Church tax (Kirchensteuer):** 8–9% of the 25% for members (24.51–24.96% effective)

**There is no holding period preference in Germany** — long-term and short-term gains are taxed identically at 26.375%.

### Freistellungsauftrag — the annual tax exemption
Each German taxpayer gets an annual exemption:
- **Single:** €1,000/year (raised from €801 in 2023)
- **Married/civil partnership:** €2,000/year

Capital income up to this threshold is completely tax-free. Strategy: harvest losses and realise gains strategically to maximise use of this allowance each year.

### Verlustverrechnung — loss offset rules
Capital losses can offset capital gains **within the same year**. Losses from equities can only offset equity gains (not interest income). Remaining losses carry forward to future years (Verlustvortrag).

### Vorabpauschale — the pre-payment tax (unique to Germany)
Introduced with the Investmentsteuerreformgesetz 2018, this is an annual pre-payment tax on unrealised gains in accumulating funds (funds that do not distribute dividends). It ensures that investors in accumulating ETFs pay at least some tax annually, even without selling.

$$\text{Vorabpauschale} = \text{NAV}_{Jan1} \times \text{Basiszins} \times 0.70 \times (1 - \text{Teilfreistellung})$$

- **Basiszins:** set annually by the German Finance Ministry (2024: 2.29%, 2023: 2.55%)
- **Teilfreistellung:** partial exemption — 30% for equity funds (>51% equities), 15% for mixed funds
- The Vorabpauschale is deducted from future capital gains when the fund is sold (avoiding double taxation)

### Teilfreistellung — partial exemption for fund gains
When you sell an ETF, only part of the gain is taxable:
- **Equity ETF (>51% equities):** 30% exempt → taxable fraction = 70%
- **Mixed fund (25–51% equities):** 15% exempt → taxable fraction = 85%
- **Bond fund (<25% equities):** 0% exempt → fully taxable

In [ ]:
# ── German tax calculations: KESt, Vorabpauschale, Teilfreistellung ──────────

def kapitalertragsteuer(gain_eur, fund_type='equity', kirchensteuer=False, kirchensatz=0.09):
    """
    Compute German capital gains tax on an ETF sale gain.
    fund_type: 'equity' (TF=30%), 'mixed' (TF=15%), 'bond' (TF=0%)
    """
    teilfreistellung = {'equity': 0.30, 'mixed': 0.15, 'bond': 0.00}.get(fund_type, 0.30)
    taxable_gain = gain_eur * (1 - teilfreistellung)  # apply partial exemption
    kest    = taxable_gain * KAPITALERTRAGSTEUER
    soli    = kest * SOLIDARITAETSZUSCHLAG
    kirche  = kest * kirchensatz if kirchensteuer else 0
    total   = kest + soli + kirche
    eff_rate = total / gain_eur if gain_eur > 0 else 0
    return {'taxable_gain': taxable_gain, 'KeSt': kest, 'Soli': soli,
            'Kirchensteuer': kirche, 'total_tax': total, 'effective_rate': eff_rate}

def vorabpauschale(nav_jan1, basiszins, fund_type='equity', actual_distribution=0):
    """
    Compute German Vorabpauschale for an accumulating ETF.
    nav_jan1: fund value on 1 January
    basiszins: Bundesbank base rate for the year
    actual_distribution: dividends actually paid out (distributing funds)
    """
    basisertrag    = nav_jan1 * basiszins * BASISERTRAG_FACTOR
    teilfreistellung = {'equity': 0.30, 'mixed': 0.15, 'bond': 0.00}.get(fund_type, 0.30)
    # Vorabpauschale = max(Basisertrag - actual distributions, 0)
    vorabpauschale_gross = max(basisertrag - actual_distribution, 0)
    taxable_vorab = vorabpauschale_gross * (1 - teilfreistellung)
    tax_vorab     = taxable_vorab * EFFECTIVE_TAX_RATE
    return {'basisertrag': basisertrag, 'vorabpauschale_gross': vorabpauschale_gross,
            'taxable': taxable_vorab, 'tax_due': tax_vorab}

# ── Example calculations
gain_scenarios = [
    ('MSCI World ETF (equity)', 5000, 'equity', False),
    ('Balanced fund (mixed)',   5000, 'mixed',  False),
    ('Gov Bond ETF (bond)',     5000, 'bond',   False),
    ('MSCI World + Kirche',    5000, 'equity', True ),
]

print('Capital gains tax breakdown on €5,000 gain:\n')
print(f'{"Fund type":<28} {"Taxable":>10} {"KeSt":>8} {"Soli":>7} {"Total tax":>10} {"Eff. rate":>10}')
print('-' * 78)
tax_data = []
for name, gain, ftype, kirche in gain_scenarios:
    r = kapitalertragsteuer(gain, ftype, kirche)
    print(f'{name:<28} €{r["taxable_gain"]:>8,.0f} €{r["KeSt"]:>7,.2f} €{r["Soli"]:>5,.2f} '
          f'€{r["total_tax"]:>8,.2f} {r["effective_rate"]*100:>9.3f}%')
    tax_data.append(r)

print('\nVorabpauschale examples (accumulating ETFs, 2024 Basiszins=2.29%):')
basiszins_2024 = 0.0229
for nav, ftype in [(10000, 'equity'), (10000, 'mixed'), (10000, 'bond')]:
    v = vorabpauschale(nav, basiszins_2024, ftype)
    print(f'  NAV=€{nav:,} {ftype:8s}: Basisertrag=€{v["basisertrag"]:.2f}  '
          f'Vorabp.=€{v["vorabpauschale_gross"]:.2f}  Tax=€{v["tax_due"]:.2f}')

# Visualise: effective tax rates across fund types and gains
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

gain_range = np.linspace(100, 20000, 100)
for ftype, col, label in [
    ('equity', PURPLE, 'Equity ETF (TF=30%)'),
    ('mixed',  TEAL,   'Mixed fund (TF=15%)'),
    ('bond',   CORAL,  'Bond ETF (TF=0%)'),
]:
    eff_rates = [kapitalertragsteuer(g, ftype)['effective_rate']*100 for g in gain_range]
    axes[0].plot(gain_range, eff_rates, color=col, lw=2, label=label)
axes[0].axhline(EFFECTIVE_TAX_RATE*100, color=GRAY, lw=1, ls='--', label=f'Max rate ({EFFECTIVE_TAX_RATE*100:.3f}%)')
axes[0].set_xlabel('Capital gain (€)')
axes[0].set_ylabel('Effective tax rate (%)')
axes[0].set_title('Effective tax rate by fund type\nTeilfreistellung reduces rate for equity funds')
axes[0].legend(fontsize=9)

# Freistellungsauftrag impact
annual_gains = np.linspace(0, 5000, 100)
tax_with_fsa    = [max(0, g - FREISTELLUNGSAUFTRAG_SINGLE) * EFFECTIVE_TAX_RATE * 0.70 for g in annual_gains]
tax_without_fsa = [g * EFFECTIVE_TAX_RATE * 0.70 for g in annual_gains]  # equity ETF
axes[1].plot(annual_gains, tax_without_fsa, color=CORAL, lw=2, label='Without Freistellungsauftrag')
axes[1].plot(annual_gains, tax_with_fsa,    color=TEAL,  lw=2, label=f'With FSA (€{FREISTELLUNGSAUFTRAG_SINGLE:,})')
axes[1].fill_between(annual_gains, tax_without_fsa, tax_with_fsa, alpha=0.2, color=TEAL, label='Tax saved by FSA')
axes[1].axvline(FREISTELLUNGSAUFTRAG_SINGLE, color=GRAY, lw=1.5, ls='--',
                label=f'FSA threshold €{FREISTELLUNGSAUFTRAG_SINGLE:,}')
axes[1].set_xlabel('Annual capital gains (€)')
axes[1].set_ylabel('Annual tax (€)')
axes[1].set_title('Freistellungsauftrag: €1,000 annual exemption\n(equity ETF, Teilfreistellung applied)')
axes[1].legend(fontsize=8)

# Vorabpauschale over different Basiszins scenarios
basiszins_history = {'2021': -0.0088, '2022': 0.0055, '2023': 0.0255, '2024': 0.0229}
nav_example = 10000
vorab_taxes_eq = []
vorab_taxes_bo = []
for yr, bz in basiszins_history.items():
    v_eq = vorabpauschale(nav_example, bz, 'equity')
    v_bo = vorabpauschale(nav_example, bz, 'bond')
    vorab_taxes_eq.append(v_eq['tax_due'])
    vorab_taxes_bo.append(v_bo['tax_due'])

x = np.arange(len(basiszins_history))
w = 0.35
axes[2].bar(x - w/2, vorab_taxes_eq, w, color=PURPLE, label='Equity ETF (TF=30%)', alpha=0.85)
axes[2].bar(x + w/2, vorab_taxes_bo, w, color=BLUE,   label='Bond ETF (TF=0%)',   alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels([f'{yr}\n(Bz={bz:.2%})' for yr, bz in basiszins_history.items()], fontsize=9)
axes[2].set_ylabel('Annual Vorabpauschale tax (€)')
axes[2].set_title(f'Vorabpauschale tax on €{nav_example:,} fund value\nVaries with annual Basiszins')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## Part 3 — Tax Lot Accounting: The Data Model

Every purchase of an ETF creates a **tax lot**: an immutable record of how many shares were bought, at what price, and on what date. When you sell, you must specify which lots you are selling — this determines the realised gain and thus the tax liability.

### The lot selection strategies

| Strategy | Rule | Pros | Cons |
|---|---|---|---|
| **FIFO** | Sell oldest lots first | Simple, regulatory default in many countries | Often tax-inefficient (oldest lots have most gain) |
| **LIFO** | Sell newest lots first | May minimise short-term gains | Not allowed in Germany for investment funds |
| **Max cost basis** | Sell highest-basis lots first | Minimises immediate tax | Requires specific identification |
| **Min cost basis** | Sell lowest-basis lots first | Accelerates loss realisation | Maximises immediate tax — used only for TLH |
| **Optimal (LP)** | Minimise tax subject to quantity constraint | Globally optimal | Requires solver, more complex |

**In Germany:** The InvStG (Investmentsteuergesetz) requires specific accounting at the fund level. For ETFs held in a custody account (Depot), FIFO is typically the default, but specific identification is generally permitted if documented.

### The lot selection as a linear program
Given $M$ lots, each with quantity $q_m$, cost basis $p_m$, and current price $P_{now}$:

$$\underset{x}{\text{minimise}} \sum_{m=1}^{M} \max(0, P_{now} - p_m) \cdot x_m$$
$$\text{subject to:} \quad \sum_{m=1}^{M} x_m = Q_{sell}, \quad 0 \leq x_m \leq q_m \; \forall m$$

This is a **linear program** — the gain per share from each lot is a constant coefficient. The greedy solution (sort by cost basis descending, take from top) is provably optimal for this unconstrained LP.

In [ ]:
# ── Tax lot data model and optimal lot selection ──────────────────────────────
@dataclass
class TaxLot:
    lot_id:        str
    asset:         str
    quantity:      float    # number of shares
    cost_basis:    float    # price per share at acquisition (€)
    acquisition:   date     # purchase date
    fund_type:     str = 'equity'

    def gain_per_share(self, current_price: float) -> float:
        return current_price - self.cost_basis

    def unrealised_return(self, current_price: float) -> float:
        return (current_price - self.cost_basis) / self.cost_basis

    def tax_if_sold(self, quantity_sold: float, current_price: float) -> float:
        gain  = max(0.0, self.gain_per_share(current_price)) * quantity_sold
        tf    = {'equity': 0.30, 'mixed': 0.15, 'bond': 0.00}.get(self.fund_type, 0.30)
        return gain * (1 - tf) * EFFECTIVE_TAX_RATE

def optimal_lot_selection(lots: List[TaxLot], shares_to_sell: float,
                           current_price: float, strategy: str = 'max_basis') -> dict:
    """
    Select lots to sell to minimise tax (default: highest basis first).
    strategy: 'max_basis' | 'fifo' | 'min_basis' | 'min_tax'
    """
    if strategy == 'max_basis':
        sorted_lots = sorted(lots, key=lambda l: -l.cost_basis)
    elif strategy == 'fifo':
        sorted_lots = sorted(lots, key=lambda l: l.acquisition)
    elif strategy == 'min_basis':
        sorted_lots = sorted(lots, key=lambda l: l.cost_basis)
    elif strategy == 'min_tax':
        sorted_lots = sorted(lots, key=lambda l: l.gain_per_share(current_price))
    else:
        sorted_lots = lots

    remaining = shares_to_sell
    selected  = []
    total_tax = 0.0
    total_gain = 0.0

    for lot in sorted_lots:
        if remaining <= 1e-9:
            break
        sell_qty = min(lot.quantity, remaining)
        tax      = lot.tax_if_sold(sell_qty, current_price)
        gain     = lot.gain_per_share(current_price) * sell_qty
        selected.append({'lot': lot, 'sell_qty': sell_qty, 'tax': tax, 'gain': gain})
        total_tax  += tax
        total_gain += max(0, gain)
        remaining  -= sell_qty

    return {'selected': selected, 'total_tax': total_tax, 'total_gain': total_gain,
            'unfilled': remaining}

# Example: 6 lots of MSCI World ETF acquired over 4 years
today = date(2024, 10, 1)
current_price_etf = 130.0

lots_msci = [
    TaxLot('L001', 'MSCI World', 100, 65.0,  date(2020,  3, 15)),  # strong gain (post-COVID low)
    TaxLot('L002', 'MSCI World', 80,  88.0,  date(2021,  6, 10)),  # good gain
    TaxLot('L003', 'MSCI World', 60,  105.0, date(2022,  4, 20)),  # moderate gain
    TaxLot('L004', 'MSCI World', 50,  118.0, date(2023,  1,  5)),  # small gain
    TaxLot('L005', 'MSCI World', 40,  128.0, date(2024,  3, 12)),  # tiny gain
    TaxLot('L006', 'MSCI World', 30,  138.0, date(2024,  8, 20)),  # unrealised LOSS
]

shares_to_sell = 120

strategies = ['max_basis', 'fifo', 'min_basis']
results = {s: optimal_lot_selection(lots_msci, shares_to_sell, current_price_etf, s) for s in strategies}

# ── Visualise ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Lot cost basis vs current price
ax = axes[0]
lot_labels = [f'L{i+1}\n({l.quantity}sh\n@€{l.cost_basis:.0f})' for i, l in enumerate(lots_msci)]
gains_pct  = [(current_price_etf - l.cost_basis)/l.cost_basis*100 for l in lots_msci]
colors_lot = [GREEN if g > 0 else RED for g in gains_pct]
bars = ax.barh(lot_labels, gains_pct, color=colors_lot, edgecolor='white', height=0.6)
ax.axvline(0, color='black', lw=1)
for bar, g, lot in zip(bars, gains_pct, lots_msci):
    ax.text(g + (0.5 if g >= 0 else -0.5), bar.get_y()+bar.get_height()/2,
            f'{g:+.0f}% | basis €{lot.cost_basis:.0f}',
            va='center', ha='left' if g >= 0 else 'right', fontsize=8)
ax.set_xlabel('Unrealised gain/loss (%)')
ax.set_title(f'Tax lots — current price €{current_price_etf:.0f}\nGreen = gain, Red = loss')

# Tax comparison by strategy
ax = axes[1]
strat_labels = ['Max basis\n(optimal)', 'FIFO\n(default)', 'Min basis\n(worst)']
taxes = [results[s]['total_tax'] for s in strategies]
gains = [results[s]['total_gain'] for s in strategies]
colors_bar = [GREEN, AMBER, RED]
bars = ax.bar(strat_labels, taxes, color=colors_bar, edgecolor='white', width=0.55)
for bar, t, g in zip(bars, taxes, gains):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
            f'Tax: €{t:.0f}\nGain: €{g:.0f}', ha='center', fontsize=9)
ax.set_ylabel('Total tax due (€)')
ax.set_title(f'Tax by lot selection strategy\n(sell {shares_to_sell} shares at €{current_price_etf})')
ax.set_ylim(0, max(taxes) * 1.35)
tax_saved = taxes[2] - taxes[0]
ax.annotate(f'Tax saved vs FIFO:\n€{taxes[1]-taxes[0]:.0f}',
            xy=(0, taxes[0]), xytext=(1.2, taxes[1]*0.5),
            fontsize=9, color=GREEN,
            arrowprops=dict(arrowstyle='->', color=GREEN))

# Show which lots are selected under max_basis
ax = axes[2]
lot_positions = np.arange(len(lots_msci))
sold_qtys = np.zeros(len(lots_msci))
for sel in results['max_basis']['selected']:
    for i, lot in enumerate(lots_msci):
        if lot.lot_id == sel['lot'].lot_id:
            sold_qtys[i] = sel['sell_qty']

kept_qtys = np.array([l.quantity for l in lots_msci]) - sold_qtys
ax.barh(lot_positions, kept_qtys, color=[GREEN if g > 0 else RED for g in gains_pct],
        alpha=0.4, height=0.6, label='Kept')
ax.barh(lot_positions, sold_qtys, left=kept_qtys,
        color=GRAY, alpha=0.9, height=0.6, hatch='///', label='Sold (max basis)')
ax.set_yticks(lot_positions)
ax.set_yticklabels([f'L{i+1} (basis €{l.cost_basis:.0f})' for i, l in enumerate(lots_msci)])
ax.set_xlabel('Shares')
ax.set_title(f'Max-basis strategy: which lots are sold?\nTax = €{results["max_basis"]["total_tax"]:.0f}')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Lot selection results (sell {shares_to_sell} shares at €{current_price_etf}):')
for s, strat_label in zip(strategies, strat_labels):
    r = results[s]
    print(f'  {s:12s}: tax = €{r["total_tax"]:7.2f}   gain = €{r["total_gain"]:7.2f}')
print(f'\nUsing max-basis vs FIFO saves: €{results["fifo"]["total_tax"]-results["max_basis"]["total_tax"]:.2f}')

---
## Part 4 — Tax-Loss Harvesting: Full Quantification

Tax-loss harvesting (TLH) is the practice of **proactively selling positions with unrealised losses** to:
1. Realise the loss as a tax asset (it offsets current-year capital gains)
2. Immediately buy a similar but not identical instrument to maintain market exposure
3. Achieve a **deferral** — you have now reset the cost basis lower, so the future gain is larger, but the tax on that larger gain is paid later

### The TLH mechanics (step by step)
1. Client holds MSCI World ETF A with €10,000 unrealised loss
2. Sell ETF A → realise €10,000 loss → add to tax loss pool
3. Immediately buy MSCI World ETF B (different provider, similar but not identical) → maintain equity exposure
4. The €10,000 loss offsets €10,000 of gains elsewhere → tax refund of €10,000 × 26.375% = €2,637.50
5. ETF B has cost basis = current price (lower than ETF A's original basis)
6. When ETF B is eventually sold, the gain is larger by exactly the harvested loss — but you've had €2,637.50 compounding for T years in the meantime

### PV of TLH benefit — the exact formula
$$\text{PV benefit} = \text{tax\_rate} \times L \times \left[1 - \frac{1}{(1+r)^T}\right] \approx \text{tax\_rate} \times L \times r \times T$$

Where $L$ = harvested loss amount, $r$ = portfolio growth rate, $T$ = years until eventual sale.

### The wash-sale rule in Germany
Germany does not have an explicit codified "wash-sale" rule equivalent to the US 30-day rule. However:
- **Steuergestaltungsmissbrauch (§42 AO):** selling and immediately repurchasing the identical instrument could be challenged as tax abuse
- **Practical rule:** use a different ETF provider's fund tracking the same index (e.g., iShares MSCI World → Vanguard FTSE All-World or Amundi MSCI World)
- The substitute must have sufficiently different structure that it is not the same investment (different index, different provider, different ISIN)

### When TLH is worth doing
TLH is valuable when:
1. The harvested loss is substantial (transaction costs must be covered)
2. There are gains to offset (otherwise the loss just carries forward)
3. The Freistellungsauftrag is already used up (harvesting within the exemption has no value)
4. The substitute instrument is sufficiently similar that tracking error vs original is minimal

In [ ]:
# ── Tax-loss harvesting: full simulation ─────────────────────────────────────
def tlh_pv_benefit(loss_amount, tax_rate, growth_rate, deferral_years, tx_cost_rate=0.0003):
    """
    Compute present value benefit of harvesting a tax loss.
    Net of transaction costs (selling + buying substitute).
    """
    tax_refund    = loss_amount * tax_rate
    tx_costs      = loss_amount * tx_cost_rate * 2   # sell + buy substitute
    pv_tax_future = tax_refund / (1 + growth_rate)**deferral_years
    pv_benefit    = tax_refund - pv_tax_future - tx_costs
    return {'pv_benefit': pv_benefit, 'tax_refund_now': tax_refund,
            'pv_tax_future': pv_tax_future, 'tx_costs': tx_costs,
            'net_benefit': pv_benefit - tx_costs,
            'break_even_loss': tx_costs / (tax_rate * (1 - 1/(1+growth_rate)**deferral_years))}

# Scan parameters
losses     = np.linspace(100, 20000, 100)
deferral_years_v = [1, 3, 5, 10, 20]
r_port     = 0.07
tx_cost    = 0.0003   # 3bps

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# PV benefit vs loss size for different deferral horizons
colors_d = [BLUE, TEAL, GREEN, AMBER, CORAL]
for dy, col in zip(deferral_years_v, colors_d):
    pvbs = [tlh_pv_benefit(L, EFFECTIVE_TAX_RATE, r_port, dy, tx_cost)['pv_benefit'] for L in losses]
    axes[0, 0].plot(losses, pvbs, color=col, lw=2, label=f'{dy}yr deferral')
axes[0, 0].axhline(0, color='black', lw=1)
axes[0, 0].set_xlabel('Harvested loss amount (€)')
axes[0, 0].set_ylabel('PV benefit of TLH (€)')
axes[0, 0].set_title(f'TLH PV benefit by loss size and deferral\n(r={r_port*100:.0f}%, tx cost={tx_cost*100:.2f}%)')
axes[0, 0].legend(fontsize=9)

# Break-even loss threshold (below which TLH is not worth it)
break_evens = [tlh_pv_benefit(1000, EFFECTIVE_TAX_RATE, r_port, dy, tx_cost)['break_even_loss']
               for dy in deferral_years_v]
axes[0, 1].bar([str(d)+'yr' for d in deferral_years_v], break_evens,
               color=colors_d, edgecolor='white', width=0.55)
for i, (dy, be) in enumerate(zip(deferral_years_v, break_evens)):
    axes[0, 1].text(i, be+5, f'€{be:.0f}', ha='center', fontsize=10, fontweight='bold')
axes[0, 1].set_xlabel('Expected deferral horizon')
axes[0, 1].set_ylabel('Minimum loss for TLH to be profitable (€)')
axes[0, 1].set_title('Break-even loss threshold\n(below this, transaction costs exceed TLH benefit)')

# Simulate annual TLH on a growing portfolio
np.random.seed(42)
n_years   = 20
V0        = 100_000
mu_ann    = 0.08
sigma_ann = 0.18
tlh_threshold = 500   # only harvest if loss > €500

# Simulate two scenarios: with and without TLH
def simulate_portfolio(V0, mu, sigma, years, tlh=False, threshold=500):
    np.random.seed(99)
    annual_rets = np.random.normal(mu - 0.5*sigma**2, sigma, years)
    V       = V0
    tax_pool= 0.0      # accumulated tax loss carryforward
    total_tax_paid = 0.0
    values  = [V0]
    taxes_paid = []

    for ret in annual_rets:
        V_prev = V
        V = V * (1 + ret)
        gain = V - V_prev

        if tlh and gain < -threshold:
            # Harvest the loss: add to loss pool, reset basis (no market impact)
            loss_harvested = abs(gain)
            tax_pool += loss_harvested * EFFECTIVE_TAX_RATE * 0.70  # equity TF
        elif gain > 0:
            taxable = max(0, gain * 0.70 * EFFECTIVE_TAX_RATE - tax_pool)
            tax_pool = max(0, tax_pool - gain * 0.70 * EFFECTIVE_TAX_RATE)
            total_tax_paid += taxable
            V -= taxable

        values.append(V)
        taxes_paid.append(total_tax_paid)

    return np.array(values), np.array(taxes_paid)

vals_no_tlh, taxes_no = simulate_portfolio(V0, mu_ann, sigma_ann, n_years, tlh=False)
vals_tlh,    taxes_yes = simulate_portfolio(V0, mu_ann, sigma_ann, n_years, tlh=True)

yrs = np.arange(n_years + 1)
axes[1, 0].plot(yrs, vals_no_tlh/1000, color=CORAL, lw=2.5, label='Without TLH')
axes[1, 0].plot(yrs, vals_tlh/1000,    color=TEAL,  lw=2.5, label='With TLH')
axes[1, 0].fill_between(yrs, vals_no_tlh/1000, vals_tlh/1000,
                          alpha=0.15, color=TEAL, label='TLH benefit (€k)')
axes[1, 0].set_xlabel('Year'); axes[1, 0].set_ylabel('Portfolio value (€k)')
axes[1, 0].set_title(f'Portfolio growth with vs without TLH\n(V0=€{V0:,}, μ={mu_ann*100:.0f}%, σ={sigma_ann*100:.0f}%)')
axes[1, 0].legend(fontsize=9)

# TLH value by Basiszins environment (low rate → less Vorabpauschale → TLH more valuable)
rate_env = np.linspace(0, 0.06, 50)
tlh_benefits_10yr = [tlh_pv_benefit(5000, EFFECTIVE_TAX_RATE, r, 10)['pv_benefit'] for r in rate_env]
axes[1, 1].plot(rate_env*100, tlh_benefits_10yr, color=PURPLE, lw=2.5)
axes[1, 1].axvline(r_port*100, color=AMBER, lw=1.5, ls='--', label=f'Assumed r={r_port*100:.0f}%')
axes[1, 1].set_xlabel('Portfolio growth rate (%)')
axes[1, 1].set_ylabel('PV benefit of TLH (€ on €5,000 loss, 10yr)')
axes[1, 1].set_title('TLH benefit grows with portfolio returns\nHigher returns → more valuable to defer tax')
axes[1, 1].legend()

plt.suptitle('Tax-loss harvesting: mechanics and quantification', y=1.01, fontweight='bold')
plt.tight_layout()
plt.show()

final_benefit = vals_tlh[-1] - vals_no_tlh[-1]
print(f'TLH simulation results after {n_years} years (V0=€{V0:,}):')
print(f'  Without TLH: €{vals_no_tlh[-1]:,.0f}')
print(f'  With TLH:    €{vals_tlh[-1]:,.0f}')
print(f'  TLH benefit: €{final_benefit:,.0f} ({final_benefit/V0*100:.1f}% of initial capital)')

---
## Part 5 — Tax-Aware Rebalancing: Integrating Tax into the Optimizer

The full rebalancing problem must include tax costs as a penalty in the objective. When you sell a position with an unrealised gain, the tax due is:
$$\text{Tax}_i(\Delta w_i) = \text{tax\_rate} \times (1-TF_i) \times \max(0, -\Delta w_i) \times V \times g_i$$

where $g_i = (P_{now} - P_{basis}) / P_{now}$ is the **embedded gain rate** (fraction of current value that is unrealised gain) of the best available lot in asset $i$, and $TF_i$ is the Teilfreistellung for that fund type.

The full tax-aware rebalancing objective:
$$\underset{\mathbf{w}}{\text{minimise}} \quad \lambda_{TE}(\mathbf{w}-\mathbf{w}^*)^\top\Sigma(\mathbf{w}-\mathbf{w}^*) + \lambda_{TC}\,\mathbf{c}^\top|\Delta\mathbf{w}| + \lambda_{tax}\sum_i \text{Tax}_i(\Delta w_i)$$

The tax term is convex because $\max(0, x)$ = `cp.pos(x)` is convex in cvxpy.

### The key trade-off
The optimizer must weigh:
- The **tracking error cost** of not rebalancing (staying in a drifted portfolio)
- The **transaction cost** of trading
- The **tax cost** of realising gains

For a highly appreciated position (large embedded gain), the optimizer will often prefer to under-rebalance — accepting some tracking error to avoid the tax crystallisation.

In [ ]:
# ── Tax-aware rebalancing optimizer ──────────────────────────────────────────
import numpy as np
from scipy.optimize import minimize

# 4-asset portfolio setup
N_assets = 4
asset_names_4 = ['MSCI World ETF', 'EM Equity ETF', 'Gov Bond ETF', 'Corp Bond ETF']
fund_types     = ['equity', 'equity', 'bond', 'mixed']
vols_4  = np.array([0.18, 0.22, 0.06, 0.10])
corr_4  = np.array([[1.00, 0.75, -0.20, 0.20],
                     [0.75, 1.00, -0.15, 0.15],
                     [-0.20,-0.15, 1.00, 0.60],
                     [0.20, 0.15, 0.60, 1.00]])
Sigma_4 = np.outer(vols_4, vols_4) * corr_4

w_curr = np.array([0.50, 0.18, 0.12, 0.20])  # drifted current weights
w_tgt  = np.array([0.35, 0.20, 0.25, 0.20])  # target weights

# Embedded gain rates by asset (fraction of current value that is unrealised gain)
# Positive = gain (tax owed on sell), Negative = loss (tax benefit on sell)
embedded_gain_rates = np.array([0.45, 0.10, -0.05, 0.25])  # MSCI World: 45% gain, EM: 10%, Bond: 5% loss, Corp: 25%
tf_rates = np.array([0.30, 0.30, 0.00, 0.15])  # Teilfreistellung per fund

V = 100_000  # portfolio value €
cost_rates = np.array([0.0003, 0.0008, 0.0002, 0.0004])  # bid-ask

def effective_tax_rate_per_asset(fund_types_list):
    """Effective tax rate accounting for Teilfreistellung."""
    tf = {'equity': 0.30, 'mixed': 0.15, 'bond': 0.00}
    return np.array([EFFECTIVE_TAX_RATE * (1 - tf[ft]) for ft in fund_types_list])

eff_tax_rates = effective_tax_rate_per_asset(fund_types)

def tax_cost_of_sell(delta_w, embedded_gains, eff_tax):
    """Tax due when selling (delta_w < 0 = selling)."""
    # Only gain-realising sells incur tax: delta_w < 0 AND embedded_gain > 0
    sell_amounts = np.maximum(-delta_w, 0)   # how much of each asset we're selling (weight units)
    gain_portion = np.maximum(embedded_gains, 0)  # only positive gains are taxable
    return np.sum(sell_amounts * gain_portion * eff_tax)

def rebal_objective(w, lambda_te, lambda_tc, lambda_tax):
    delta_w = w - w_curr
    te_term  = lambda_te  * (w - w_tgt) @ Sigma_4 @ (w - w_tgt)
    tc_term  = lambda_tc  * np.sum(cost_rates * np.abs(delta_w))
    tax_term = lambda_tax * tax_cost_of_sell(delta_w, embedded_gain_rates, eff_tax_rates)
    return te_term + tc_term + tax_term

constraints = [{'type': 'eq',   'fun': lambda w: w.sum() - 1},
               {'type': 'ineq', 'fun': lambda w: 0.30 - np.abs(w - w_curr).sum()}]
bounds = [(0.0, 0.50)] * N_assets

lambda_te  = 10.0
lambda_tc  = 1.0
lambda_tax_range = [0, 0.5, 1.0, 2.0, 5.0, 10.0]  # sweep tax penalty

results_tax = []
for lt in lambda_tax_range:
    res = minimize(rebal_objective, w_curr, args=(lambda_te, lambda_tc, lt),
                   method='SLSQP', bounds=bounds, constraints=constraints)
    w_opt = res.x
    delta = w_opt - w_curr
    te    = np.sqrt((w_opt-w_tgt)@Sigma_4@(w_opt-w_tgt))*100
    tc    = np.sum(cost_rates * np.abs(delta)) * 100
    tax   = tax_cost_of_sell(delta, embedded_gain_rates, eff_tax_rates) * 100
    results_tax.append({'lt': lt, 'w': w_opt, 'delta': delta, 'te': te, 'tc': tc, 'tax': tax})

# ── Visualise ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# TE vs tax penalty
ax = axes[0, 0]
te_vals  = [r['te']  for r in results_tax]
tax_vals = [r['tax'] for r in results_tax]
ax.plot(lambda_tax_range, te_vals,  'o-', color=PURPLE, lw=2, ms=8, label='Post-trade TE (%)')
te_pre = np.sqrt((w_curr-w_tgt)@Sigma_4@(w_curr-w_tgt))*100
ax.axhline(te_pre, color=CORAL, lw=1.5, ls='--', label=f'Pre-trade TE = {te_pre:.2f}%')
ax2 = ax.twinx()
ax2.plot(lambda_tax_range, tax_vals, 's-', color=AMBER, lw=2, ms=8, label='Tax crystallised (%)')
ax2.set_ylabel('Tax crystallised (% portfolio)', color=AMBER)
ax.set_xlabel('Tax penalty λ_tax')
ax.set_ylabel('Post-trade TE (%)', color=PURPLE)
ax.set_title('Higher tax penalty → accept more TE to avoid tax\ntax-averse optimizer holds appreciated positions')
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labs1+labs2, fontsize=8)

# Optimal weights for low vs high tax penalty
ax = axes[0, 1]
r_low  = results_tax[0]   # lambda_tax = 0 (ignore tax)
r_high = results_tax[-1]  # lambda_tax = 10 (very tax-averse)
x = np.arange(N_assets)
w2 = 0.2
ax.bar(x - w2*1.5, w_curr*100,     w2, color=GRAY,   label='Current', alpha=0.85)
ax.bar(x - w2*0.5, w_tgt*100,      w2, color=GREEN,  label='Target', alpha=0.85)
ax.bar(x + w2*0.5, r_low['w']*100, w2, color=BLUE,   label=f'Optimal λ_tax=0 (tax blind)', alpha=0.85)
ax.bar(x + w2*1.5, r_high['w']*100,w2, color=CORAL,  label=f'Optimal λ_tax=10 (tax-averse)', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(asset_names_4, rotation=15, fontsize=9)
ax.set_ylabel('Weight (%)')
ax.set_title('Tax-averse optimizer holds appreciated positions\n(MSCI World has 45% embedded gain → avoid selling)')
ax.legend(fontsize=8)

# Embedded gain rates and how they affect trades
ax = axes[1, 0]
colors_gain = [GREEN if g > 0 else RED for g in embedded_gain_rates]
bars = ax.bar(asset_names_4, embedded_gain_rates*100, color=colors_gain, edgecolor='white', width=0.55)
ax.axhline(0, color='black', lw=1)
for bar, g, eff in zip(bars, embedded_gain_rates, eff_tax_rates):
    tax_pct = max(0, g) * eff * 100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height() + (1 if g >= 0 else -3),
            f'{g*100:+.0f}%\ntax cost\n{tax_pct:.1f}%',
            ha='center', fontsize=8)
ax.set_ylabel('Embedded gain rate (%)')
ax.set_title('Embedded gain rates by asset\nGreen = gain (sell = tax cost), Red = loss (sell = tax benefit)')

# Three-way trade-off: TE, TC, Tax
ax = axes[1, 1]
r_no_tax = results_tax[0]
r_mid    = results_tax[3]
r_max    = results_tax[-1]
categories = ['Tracking\nError (%)', 'Transaction\nCost (%)', 'Tax\nCost (%)']
vals_no   = [r_no_tax['te'], r_no_tax['tc']*10, r_no_tax['tax']*10]
vals_mid  = [r_mid['te'],    r_mid['tc']*10,    r_mid['tax']*10]
vals_max  = [r_max['te'],    r_max['tc']*10,    r_max['tax']*10]
x3 = np.arange(3); w3 = 0.25
ax.bar(x3 - w3, vals_no,  w3, color=BLUE,   label='λ_tax=0 (tax-blind)',   alpha=0.85)
ax.bar(x3,      vals_mid, w3, color=AMBER,  label='λ_tax=2 (balanced)',    alpha=0.85)
ax.bar(x3 + w3, vals_max, w3, color=CORAL,  label='λ_tax=10 (tax-averse)', alpha=0.85)
ax.set_xticks(x3); ax.set_xticklabels(categories)
ax.set_ylabel('Cost (scaled for comparison)')
ax.set_title('Three-way trade-off in tax-aware rebalancing\nMinimising one cost increases others')
ax.legend(fontsize=9)

plt.suptitle('Tax-aware rebalancing: integrating tax into the convex optimizer', y=1.01, fontweight='bold')
plt.tight_layout()
plt.show()

print('Optimal trades (tax-blind vs tax-averse):')
print(f'{"Asset":<18} {"Current":>8} {"Target":>8} {"λ_tax=0":>9} {"λ_tax=10":>10} {"Gain rate":>10}')
print('-' * 75)
for i, name in enumerate(asset_names_4):
    print(f'{name:<18} {w_curr[i]*100:>7.1f}% {w_tgt[i]*100:>7.1f}% '
          f'{results_tax[0]["w"][i]*100:>8.1f}% {results_tax[-1]["w"][i]*100:>9.1f}% '
          f'{embedded_gain_rates[i]*100:>9.0f}%')

---
## Part 6 — Pan-European Tax Landscape

Scalable Capital operates across Germany, Austria, UK, Italy, France, and Spain. Each jurisdiction has different tax rules — building a pan-European tax engine is precisely what the JD calls out.

| Country | Tax rate | Holding period preference | Loss offset | Annual exemption |
|---|---|---|---|---|
| **Germany** | 26.375% flat | None | Same year, equity vs equity only | €1,000 |
| **Austria** | 27.5% flat (25% for bank interest) | None | Same year | None |
| **UK** | 10% (basic rate) / 20% (higher) | None since 2024 | 3yr carryback, unlimited forward | £3,000 CGT exemption |
| **Italy** | 26% flat | None | 4yr carryforward | None |
| **France** | 30% PFU (flat) or progressive barème | None | 10yr carryforward | None |
| **Spain** | 19-26% progressive (savings income) | None | 4yr carryforward | None |

The key architectural implication: the tax engine must be **country-aware** — the lot selection algorithm, the TLH threshold, the Vorabpauschale calculation, and the reporting format all differ by jurisdiction.

In [ ]:
# ── Pan-European tax comparison on identical gain ─────────────────────────────

# Tax rules per country
eu_tax_rules = {
    'Germany (DE)':  {'rate_low': 0.26375, 'rate_high': 0.26375, 'threshold': 0,       'exemption': 1000,  'fund_tf': 0.30},
    'Austria (AT)':  {'rate_low': 0.275,   'rate_high': 0.275,   'threshold': 0,       'exemption': 0,     'fund_tf': 0.00},
    'UK (GB)':       {'rate_low': 0.10,    'rate_high': 0.20,    'threshold': 3000,    'exemption': 3000,  'fund_tf': 0.00},
    'Italy (IT)':    {'rate_low': 0.26,    'rate_high': 0.26,    'threshold': 0,       'exemption': 0,     'fund_tf': 0.00},
    'France (FR)':   {'rate_low': 0.30,    'rate_high': 0.30,    'threshold': 0,       'exemption': 0,     'fund_tf': 0.00},
    'Spain (ES)':    {'rate_low': 0.19,    'rate_high': 0.26,    'threshold': 6000,    'exemption': 0,     'fund_tf': 0.00},
}

def compute_tax_eu(country_rules, gain_eur, is_equity_fund=True, is_high_rate=False):
    r = country_rules
    rate = r['rate_high'] if (is_high_rate and gain_eur > r['threshold']) else r['rate_low']
    exempt = min(gain_eur, r['exemption'])
    taxable = max(0, gain_eur - exempt) * (1 - r['fund_tf'] if is_equity_fund else 1)
    return taxable * rate

gains_test = [1000, 5000, 20000, 100000]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Tax due by country for different gain sizes
countries = list(eu_tax_rules.keys())
colors_eu = [PURPLE, BLUE, CORAL, TEAL, AMBER, GREEN]
x = np.arange(len(gains_test))
w_bar = 0.13
for i, (country, rules) in enumerate(eu_tax_rules.items()):
    taxes = [compute_tax_eu(rules, g, is_equity_fund=True) for g in gains_test]
    offset = (i - len(eu_tax_rules)/2) * w_bar
    axes[0].bar(x + offset, taxes, w_bar, color=colors_eu[i], label=country, alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'€{g:,}' for g in gains_test])
axes[0].set_xlabel('Capital gain amount')
axes[0].set_ylabel('Tax due (€)')
axes[0].set_title('Tax due on equity ETF gain by country\n(same gain, very different tax bills!)')
axes[0].legend(fontsize=8)

# Effective tax rates
gain_range = np.linspace(100, 50000, 200)
for country, rules, col in zip(countries, eu_tax_rules.values(), colors_eu):
    eff_rates = [compute_tax_eu(rules, g, True)/g*100 for g in gain_range]
    axes[1].plot(gain_range, eff_rates, color=col, lw=2, label=country)
axes[1].set_xlabel('Capital gain (€)')
axes[1].set_ylabel('Effective tax rate (%)')
axes[1].set_title('Effective tax rate by gain size and country\nExemptions cause non-monotonic rates at low gains')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Tax on €10,000 equity ETF gain by country:')
for country, rules in eu_tax_rules.items():
    tax = compute_tax_eu(rules, 10000, True)
    eff = tax/10000*100
    bar = '█' * int(eff)
    print(f'  {country:16s}: €{tax:,.2f}  ({eff:.1f}%)  {bar}')

---
## Summary: Tax Optimization Mental Map

| Concept | What it is | Key formula | Production implementation |
|---|---|---|---|
| KeSt | German 26.375% flat CGT | `gain × (1-TF) × 26.375%` | Applied at lot sale, deducted by broker |
| Teilfreistellung | Partial exemption | equity: 30%, mixed: 15%, bond: 0% | Per-fund metadata flag |
| Freistellungsauftrag | Annual €1,000 exemption | First €1,000 gains tax-free | Track annual usage per client |
| Vorabpauschale | Annual pre-payment | `NAV × Basiszins × 0.70 × (1-TF)` | Run annually in January batch |
| Lot selection | Which shares to sell | Sort by cost basis descending | Greedy O(M log M), LP for constraints |
| TLH benefit | Deferral PV | `tax × L × r × T` | Threshold scan daily, harvest if >€500 |
| Tax-aware opt | Penalty in QP | Add `λ_tax × Σ max(0,−Δw)×g×τ` | cvxpy `cp.pos()` term |

**Three things to say in any interview about tax optimization:**
1. Tax optimization in wealth management is about **deferral, not avoidance** — every euro deferred earns more compound return
2. Lot selection is an $O(M\log M)$ greedy algorithm (sort by cost basis descending) — provably optimal for the unconstrained case
3. Germany's **Vorabpauschale** makes pan-European tax engines non-trivial — each country has unique rules that require a country-aware strategy layer on top of the core optimizer